###Process Addresses Data
- Ingest the data into the data lakehouse- bronze-addresses
- Perform DQ check and transform the data as required- silver-Addresses_clean
- Apply chnages to the addressess data (SCD-2) - silver_addresses

####1. Ingest the data into the data lakehouse- bronze-addresses

![image_1787937190633.png](./image_1787937190633.png "image_1787937190633.png")

In [0]:
from pyspark import pipelines as dp
import pyspark.sql.functions as F


In [0]:

@dp.table(
    name='bronze_addresses',
    table_properties={'quality': 'bronze'},
    comment='Raw addresses data ingested from the source system'
)
def create_bronze_addresses():
    return (
        spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('cloudFiles.inferColumnTypes', 'true')
        .load('/Volumes/circuitbox/landing/operational_data/addresses')
        .select(
            '*',
            F.col('_metadata.file_path').alias('input_file_path'),
            F.current_timestamp().alias('ingest_timestamp')
        )
    )


####2. Perform DQ check and transform the data as required- silver-Addresses_clean

![image_1787939134801.png](./image_1787939134801.png "image_1787939134801.png")

In [0]:
@dp.table(
    name='bronze_addresses_clean',
    table_properties={'quality': 'silver'},
    comment='Cleaned addresses data ingested in the silver layer'
)
@dp.expect_or_fail('valid_customer_id', 'customer_id IS NOT NULL')
@dp.expect_or_drop('valid_address', 'address_line_1 IS NOT NULL')
@dp.expect('valid_postcode', 'length(postcode) = 5')
def create_silver_addresses_clean():
    return (
        spark.readStream.table('LIVE.bronze_addresses')
        .select(
            "customer_id",
            "address_line_1",
            "city",
            "state",
            "postcode",
            F.col('created_date').cast('date')
        )
    
    )


####3. Apply changes to the addressess data (SCD-2) - silver_addresses

In [0]:
dp.create_streaming_table(
    name='silver_addresses',
    table_properties={'quality': 'silver'},
    comment='SCD Type2 addressess data'
);


In [0]:
dp.apply_changes(
    target='silver_addresses',
    source='bronze_addresses_clean',
    sequence_by='created_date',
    keys=['customer_id'],
    stored_as_scd_type=2
)